# LeetCode 886 — Possible Bipartition bằng Python

Link đề: https://leetcode.com/problems/possible-bipartition/ · Độ khó: **Medium** · Chủ đề: DFS, BFS, Union Find (tập hợp rời nhau), Graph (đồ thị), Graph Coloring (tô màu đồ thị), Bipartite Graph (đồ thị hai phía)

Skeleton LeetCode:

```python
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        ...
```

Ghi chú: mỗi cặp "ghét nhau" là một **cạnh**; hai người ghét nhau **không được cùng nhóm**. Chia được thành hai nhóm khi và chỉ khi đồ thị là **đồ thị hai phía** (bipartite), tức là có thể **tô hai màu** sao cho hai đầu mỗi cạnh khác màu. Đồ thị có thể **không liên thông** nên phải thử từng thành phần.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia `n` người (nhãn `1..n`) thành hai nhóm (kích thước tùy ý, có thể rỗng) sao cho không cặp ghét nhau nào cùng nhóm.
2. Nắm các công cụ Python cần thiết: danh sách kề `g = [[] for _ in range(n + 1)]`, `collections.deque`, mảng màu `color = [0] * (n + 1)` (0 là chưa tô, 1 và 2 là hai nhóm, `3 - c` đổi màu), ngăn xếp tường minh thay cho đệ quy, và cấu trúc Union Find có nén đường.
3. Hiểu nguyên lý: **đồ thị là hai phía ⇔ không có chu trình độ dài lẻ**; thuật toán tô màu thất bại đúng ở một cạnh nối hai đỉnh cùng màu.
4. Đi qua 4 cách: thử mọi cách chia (brute), tô màu bằng DFS, tô màu bằng BFS, và Union Find kiểu "kẻ thù của kẻ thù là bạn"; đối chiếu bằng oracle vét cạn, đồ thị sinh có chủ đích (biết trước đáp án) và bằng chứng (nhóm chia được kiểm tra trực tiếp).
5. Tránh các lỗi kinh điển: nhãn bắt đầu từ `1`, chỉ chạy DFS từ một đỉnh (đồ thị không liên thông), tô màu mà không kiểm tra xung đột, và đệ quy quá sâu.

## 2. Đề bài & ví dụ

Ta muốn chia `n` người (nhãn từ `1` đến `n`) thành hai nhóm với **kích thước bất kỳ**. Mỗi người có thể ghét một số người khác, và họ không được ở cùng nhóm.

Cho số nguyên `n` và mảng `dislikes` trong đó `dislikes[i] = [ai, bi]` nghĩa là người `ai` không ưa người `bi`. Trả về `true` nếu có thể chia mọi người thành hai nhóm như vậy, ngược lại trả `false`.

**Ví dụ 1**

- Input: `n = 4`, `dislikes = [[1,2],[1,3],[2,4]]`
- Output: `true`
- Giải thích: nhóm thứ nhất `[1, 4]`, nhóm thứ hai `[2, 3]`.

**Ví dụ 2**

- Input: `n = 3`, `dislikes = [[1,2],[1,3],[2,3]]`
- Output: `false`
- Giải thích: ba người đôi một ghét nhau cần ít nhất 3 nhóm.

**Ràng buộc:**

- `1 <= n <= 2000`
- `0 <= dislikes.length <= 10^4`
- `dislikes[i].length == 2`
- `1 <= ai < bi <= n`
- Mọi cặp trong `dislikes` là **duy nhất**.

In [1]:
from typing import List
from collections import deque
from itertools import product
import random, sys, time

examples = [
    (4, [[1, 2], [1, 3], [2, 4]], True),
    (3, [[1, 2], [1, 3], [2, 3]], False),
]
for n, dislikes, expected in examples:
    print(f"n={n} dislikes={dislikes} kỳ vọng={expected}")

n=4 dislikes=[[1, 2], [1, 3], [2, 4]] kỳ vọng=True
n=3 dislikes=[[1, 2], [1, 3], [2, 3]] kỳ vọng=False


## 3. Kiến thức nền

### 3.1. Danh sách kề

Mỗi cặp `[a, b]` là một cạnh **vô hướng** (ghét lẫn nhau theo nghĩa không được cùng nhóm), nên thêm `b` vào danh sách của `a` và `a` vào danh sách của `b`. Nhãn từ `1` nên mảng cần `n + 1` phần tử (bỏ phần tử `0`).

In [2]:
def build_graph(n: int, dislikes: List[List[int]]) -> List[List[int]]:
    g = [[] for _ in range(n + 1)]
    for a, b in dislikes:
        g[a].append(b)
        g[b].append(a)
    return g

g = build_graph(4, [[1, 2], [1, 3], [2, 4]])
print(g)

[[], [2, 3], [1, 4], [1], [2]]


### 3.2. Tô hai màu

Dùng mảng `color` với `0` là chưa tô, `1` và `2` là hai nhóm. Nếu một đỉnh màu `c` thì mọi đỉnh kề phải có màu `3 - c` (`3 - 1 = 2`, `3 - 2 = 1`). Một cạnh nối hai đỉnh **cùng màu** là một mâu thuẫn: không thể chia.

In [3]:
for c in (1, 2):
    print(f"màu {c} -> màu đối lập {3 - c}")

màu 1 -> màu đối lập 2
màu 2 -> màu đối lập 1


### 3.3. Hai phía ⇔ không có chu trình lẻ

Đi dọc một chu trình thì màu đổi qua mỗi cạnh; sau khi đi hết chu trình ta phải trở về màu ban đầu, nên độ dài chu trình phải **chẵn**. Ngược lại nếu mọi chu trình đều chẵn thì tô màu theo khoảng cách (chẵn/lẻ) từ gốc của từng thành phần liên thông luôn hợp lệ. Tam giác (3 người đôi một ghét nhau) là chu trình lẻ ngắn nhất.

### 3.4. Đồ thị không liên thông

Hai người không liên quan đến nhau thuộc các **thành phần liên thông** khác nhau; mỗi thành phần được tô độc lập. Phải bắt đầu tô từ **mọi** đỉnh chưa tô, không chỉ từ đỉnh `1`.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `n = 4`, cạnh `1-2`, `1-3`, `2-4`.

- Đặt `1` màu 1. Hai láng giềng `2` và `3` phải màu 2.
- `2` màu 2 nên láng giềng `4` phải màu 1.
- Kiểm tra: `1-2` (1,2), `1-3` (1,2), `2-4` (2,1): đều khác màu ⇒ `true`, nhóm `{1,4}` và `{2,3}`.

Ví dụ 2: cạnh `1-2`, `1-3`, `2-3`. Đặt `1` màu 1, `2` và `3` đều màu 2 nhưng `2-3` là cạnh nối hai đỉnh cùng màu ⇒ mâu thuẫn ⇒ `false`.

## 5. Cách 1 — Brute Force: thử mọi cách chia

**Ý tưởng.** Mỗi người thuộc nhóm `0` hoặc `1`: có `2^n` cách gán. Với mỗi cách kiểm tra mọi cạnh có hai đầu khác nhóm không. Đây là định nghĩa trực tiếp nên cũng là **oracle** cho `n` nhỏ.

In [4]:
def bipartition_brute(n: int, dislikes: List[List[int]]) -> bool:
    for assign in product((0, 1), repeat=n):
        if all(assign[a - 1] != assign[b - 1] for a, b in dislikes):
            return True
    return False

for n, dislikes, expected in examples:
    got = bipartition_brute(n, dislikes)
    print(n, dislikes, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

4 [[1, 2], [1, 3], [2, 4]] -> True OK
3 [[1, 2], [1, 3], [2, 3]] -> False OK


**Độ phức tạp và nút thắt.** `O(2^n * m)`: với `n = 2000` là không thể. Xem thời gian tăng theo `n` (đồ thị đường đi, luôn chia được nên phải duyệt tới khi tìm thấy; ta dùng đồ thị **không chia được** để vét hết):

In [5]:
for n in (10, 14, 16, 18, 20):
    dislikes = [[i, i + 1] for i in range(1, n)] + [[1, 3]]      # có tam giác 1-2-3: không chia được
    t0 = time.perf_counter()
    r = bipartition_brute(n, dislikes)
    print(f"n={n}: {1000 * (time.perf_counter() - t0):9.1f} ms (kết quả {r})")

n=10:       0.3 ms (kết quả False)
n=14:       4.8 ms (kết quả False)
n=16:      19.4 ms (kết quả False)
n=18:      76.8 ms (kết quả False)


n=20:     376.9 ms (kết quả False)


Thời gian tăng gấp đôi mỗi khi thêm một người. Nút thắt: ta thử các cách chia **không liên quan** nhau trong khi việc chia một người **quyết định** việc chia các láng giềng của họ.

## 6. Ý tưởng thuật toán / bước đột phá

**Việc chọn nhóm của một người ép buộc nhóm của mọi láng giềng.** Nếu `a` thuộc nhóm X và `a` ghét `b` thì `b` **bắt buộc** thuộc nhóm đối lập. Trong mỗi thành phần liên thông, chỉ cần chọn tùy ý nhóm của **một** đỉnh; phần còn lại bị ép hoàn toàn. Vì vậy chỉ cần **lan truyền** màu từ một đỉnh và kiểm tra xung đột, thời gian `O(n + m)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Lan truyền màu bằng **DFS** (cài bằng ngăn xếp tường minh; nộp) |
| 3 | Lan truyền màu bằng **BFS** (hàng đợi) |
| 4 | **Union Find**: tất cả kẻ thù của một người phải cùng nhóm với nhau |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tô màu bằng DFS

**Giải thích từng đoạn.**

- `g`: danh sách kề; `color = [0] * (n + 1)`.
- Với mỗi `s` từ `1` đến `n` chưa tô: tô `color[s] = 1`, đẩy vào ngăn xếp.
- Lấy `u` ra; với mỗi láng giềng `v`: nếu `color[v] == 0` thì tô `3 - color[u]` và đẩy `v`; nếu `color[v] == color[u]` thì **mâu thuẫn**: trả `False`.
- Hết mọi đỉnh mà không mâu thuẫn: `True`.

Dùng **ngăn xếp tường minh** thay vì đệ quy vì `n` tới `2000` có thể làm chuỗi đệ quy dài hơn giới hạn mặc định `1000` của Python (xem lỗi 5).

In [6]:
def bipartition_dfs(n: int, dislikes: List[List[int]]) -> bool:
    g = build_graph(n, dislikes)
    color = [0] * (n + 1)
    for s in range(1, n + 1):
        if color[s]:
            continue
        color[s] = 1
        stack = [s]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if color[v] == 0:
                    color[v] = 3 - color[u]
                    stack.append(v)
                elif color[v] == color[u]:
                    return False
    return True

for n, dislikes, expected in examples:
    got = bipartition_dfs(n, dislikes)
    print(n, dislikes, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

4 [[1, 2], [1, 3], [2, 4]] -> True OK
3 [[1, 2], [1, 3], [2, 3]] -> False OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        g = [[] for _ in range(n + 1)]
        for a, b in dislikes:
            g[a].append(b)
            g[b].append(a)
        color = [0] * (n + 1)
        for s in range(1, n + 1):
            if color[s]:
                continue
            color[s] = 1
            stack = [s]
            while stack:
                u = stack.pop()
                for v in g[u]:
                    if color[v] == 0:
                        color[v] = 3 - color[u]
                        stack.append(v)
                    elif color[v] == color[u]:
                        return False
        return True

sol_dfs = Solution().possibleBipartition
assert [sol_dfs(n, d) for n, d, _ in examples] == [e for _, _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n + m)` thời gian, `O(n + m)` bộ nhớ (danh sách kề, màu, ngăn xếp).

---

### 7.2. Cách 3 — Tô màu bằng BFS

**Ý tưởng khác.** Thay ngăn xếp bằng `deque` (hàng đợi): duyệt theo từng lớp khoảng cách; cùng cơ chế tô `3 - color[u]` và phát hiện xung đột. Màu của đỉnh bằng độ chẵn/lẻ của khoảng cách tới gốc.

In [8]:
def bipartition_bfs(n: int, dislikes: List[List[int]]) -> bool:
    g = build_graph(n, dislikes)
    color = [0] * (n + 1)
    for s in range(1, n + 1):
        if color[s]:
            continue
        color[s] = 1
        dq = deque([s])
        while dq:
            u = dq.popleft()
            for v in g[u]:
                if color[v] == color[u]:
                    return False
                if color[v] == 0:
                    color[v] = 3 - color[u]
                    dq.append(v)
    return True

for n, dislikes, expected in examples:
    assert bipartition_bfs(n, dislikes) == expected
print("BFS OK")

BFS OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        from collections import deque
        g = [[] for _ in range(n + 1)]
        for a, b in dislikes:
            g[a].append(b)
            g[b].append(a)
        color = [0] * (n + 1)
        for s in range(1, n + 1):
            if color[s]:
                continue
            color[s] = 1
            dq = deque([s])
            while dq:
                u = dq.popleft()
                for v in g[u]:
                    if color[v] == color[u]:
                        return False
                    if color[v] == 0:
                        color[v] = 3 - color[u]
                        dq.append(v)
        return True

sol_bfs = Solution().possibleBipartition
assert [sol_bfs(n, d) for n, d, _ in examples] == [e for _, _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n + m)` thời gian, `O(n + m)` bộ nhớ.

---

### 7.3. Cách 4 — Union Find: "kẻ thù của kẻ thù là bạn"

**Ý tưởng khác.** Nếu `x` ghét `y1` và `y2` thì `y1`, `y2` **đều khác nhóm với `x`**, nên `y1` và `y2` **cùng nhóm**. Với mỗi người `x` có danh sách kẻ thù `g[x]`:

- nếu `find(x) == find(y)` với một kẻ thù `y` thì `x` và `y` bị buộc cùng nhóm nhưng lại ghét nhau: **mâu thuẫn**, trả `False`;
- gộp tất cả kẻ thù của `x` vào cùng một tập: `union(g[x][0], y)` với mọi `y` trong `g[x]`.

**Giải thích từng đoạn.**

- `parent[i] = i`; `find` có **nén đường** (`parent[x] = parent[parent[x]]`).
- Thứ tự xử lý các `x` không quan trọng: mỗi `x` chỉ thêm các ràng buộc "kẻ thù của `x` cùng nhóm" và kiểm tra mâu thuẫn với trạng thái hiện tại; nếu có mâu thuẫn ở trạng thái hiện tại thì cũng mâu thuẫn ở trạng thái cuối, và ngược lại (mô tả ở mục 10).

In [10]:
def bipartition_dsu(n: int, dislikes: List[List[int]]) -> bool:
    g = build_graph(n, dislikes)
    parent = list(range(n + 1))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for x in range(1, n + 1):
        for y in g[x]:
            if find(x) == find(y):
                return False
            parent[find(y)] = find(g[x][0])
    return True

for n, dislikes, expected in examples:
    assert bipartition_dsu(n, dislikes) == expected
print("Union Find OK")

Union Find OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        g = [[] for _ in range(n + 1)]
        for a, b in dislikes:
            g[a].append(b)
            g[b].append(a)
        parent = list(range(n + 1))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for x in range(1, n + 1):
            for y in g[x]:
                if find(x) == find(y):
                    return False
                parent[find(y)] = find(g[x][0])
        return True

sol_dsu = Solution().possibleBipartition
assert [sol_dsu(n, d) for n, d, _ in examples] == [e for _, _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O((n + m) * alpha(n))` thời gian (`alpha` là hàm Ackermann ngược, gần hằng số), `O(n + m)` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 2 (DFS) cho ví dụ 1: `n = 4`, `dislikes = [[1,2],[1,3],[2,4]]`

In [12]:
def trace_dfs(n, dislikes):
    g = build_graph(n, dislikes)
    print("danh sách kề:", {i: g[i] for i in range(1, n + 1)})
    color = [0] * (n + 1)
    for s in range(1, n + 1):
        if color[s]:
            continue
        color[s] = 1
        print(f"bắt đầu thành phần mới từ {s}, tô màu 1")
        stack = [s]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if color[v] == 0:
                    color[v] = 3 - color[u]
                    stack.append(v)
                    print(f"   {u}(màu {color[u]}) -> {v}: chưa tô, tô màu {color[v]}")
                elif color[v] == color[u]:
                    print(f"   {u}(màu {color[u]}) -> {v}(màu {color[v]}): CÙNG MÀU, mâu thuẫn")
                    return False
                else:
                    print(f"   {u}(màu {color[u]}) -> {v}(màu {color[v]}): khác màu, ổn")
    print("màu cuối:", color[1:])
    return True

assert trace_dfs(4, [[1, 2], [1, 3], [2, 4]]) is True

danh sách kề: {1: [2, 3], 2: [1, 4], 3: [1], 4: [2]}
bắt đầu thành phần mới từ 1, tô màu 1
   1(màu 1) -> 2: chưa tô, tô màu 2
   1(màu 1) -> 3: chưa tô, tô màu 2
   3(màu 2) -> 1(màu 1): khác màu, ổn
   2(màu 2) -> 1(màu 1): khác màu, ổn
   2(màu 2) -> 4: chưa tô, tô màu 1
   4(màu 1) -> 2(màu 2): khác màu, ổn
màu cuối: [1, 2, 2, 1]


### 8.2. Cách 2 cho ví dụ 2 (tam giác)

In [13]:
assert trace_dfs(3, [[1, 2], [1, 3], [2, 3]]) is False

danh sách kề: {1: [2, 3], 2: [1, 3], 3: [1, 2]}
bắt đầu thành phần mới từ 1, tô màu 1
   1(màu 1) -> 2: chưa tô, tô màu 2
   1(màu 1) -> 3: chưa tô, tô màu 2
   3(màu 2) -> 1(màu 1): khác màu, ổn
   3(màu 2) -> 2(màu 2): CÙNG MÀU, mâu thuẫn


Với tam giác: `1` màu 1, đẩy `3` và `2` màu 2 (thứ tự do ngăn xếp), rồi khi xét một trong hai đỉnh màu 2 thấy láng giềng cùng màu 2 ⇒ mâu thuẫn.

### 8.3. Cách 4 (Union Find) cho ví dụ 1

In [14]:
def trace_dsu(n, dislikes):
    g = build_graph(n, dislikes)
    parent = list(range(n + 1))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for x in range(1, n + 1):
        for y in g[x]:
            print(f"x={x}, kẻ thù y={y}: find(x)={find(x)}, find(y)={find(y)}", end=" ")
            if find(x) == find(y):
                print("-> BẰNG NHAU: mâu thuẫn")
                return False
            parent[find(y)] = find(g[x][0])
            print(f"-> gộp y vào tập của kẻ thù đầu tiên {g[x][0]}; parent = {parent[1:]}")
    return True

assert trace_dsu(4, [[1, 2], [1, 3], [2, 4]]) is True
print()
assert trace_dsu(3, [[1, 2], [1, 3], [2, 3]]) is False

x=1, kẻ thù y=2: find(x)=1, find(y)=2 -> gộp y vào tập của kẻ thù đầu tiên 2; parent = [1, 2, 3, 4]
x=1, kẻ thù y=3: find(x)=1, find(y)=3 -> gộp y vào tập của kẻ thù đầu tiên 2; parent = [1, 2, 2, 4]
x=2, kẻ thù y=1: find(x)=2, find(y)=1 -> gộp y vào tập của kẻ thù đầu tiên 1; parent = [1, 2, 2, 4]
x=2, kẻ thù y=4: find(x)=2, find(y)=4 -> gộp y vào tập của kẻ thù đầu tiên 1; parent = [1, 2, 2, 1]
x=3, kẻ thù y=1: find(x)=2, find(y)=1 -> gộp y vào tập của kẻ thù đầu tiên 1; parent = [1, 2, 2, 1]
x=4, kẻ thù y=2: find(x)=1, find(y)=2 -> gộp y vào tập của kẻ thù đầu tiên 2; parent = [1, 2, 2, 1]

x=1, kẻ thù y=2: find(x)=1, find(y)=2 -> gộp y vào tập của kẻ thù đầu tiên 2; parent = [1, 2, 3]
x=1, kẻ thù y=3: find(x)=1, find(y)=3 -> gộp y vào tập của kẻ thù đầu tiên 2; parent = [1, 2, 2]
x=2, kẻ thù y=1: find(x)=2, find(y)=1 -> gộp y vào tập của kẻ thù đầu tiên 1; parent = [1, 2, 2]
x=2, kẻ thù y=3: find(x)=2, find(y)=2 -> BẰNG NHAU: mâu thuẫn


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (tô màu bằng DFS với ngăn xếp tường minh): trực quan, `O(n + m)`, không lo giới hạn đệ quy.

```python
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        g = [[] for _ in range(n + 1)]
        for a, b in dislikes:
            g[a].append(b)
            g[b].append(a)
        color = [0] * (n + 1)
        for s in range(1, n + 1):
            if color[s]:
                continue
            color[s] = 1
            stack = [s]
            while stack:
                u = stack.pop()
                for v in g[u]:
                    if color[v] == 0:
                        color[v] = 3 - color[u]
                        stack.append(v)
                    elif color[v] == color[u]:
                        return False
        return True
```

In [15]:
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        g = [[] for _ in range(n + 1)]
        for a, b in dislikes:
            g[a].append(b)
            g[b].append(a)
        color = [0] * (n + 1)
        for s in range(1, n + 1):
            if color[s]:
                continue
            color[s] = 1
            stack = [s]
            while stack:
                u = stack.pop()
                for v in g[u]:
                    if color[v] == 0:
                        color[v] = 3 - color[u]
                        stack.append(v)
                    elif color[v] == color[u]:
                        return False
        return True

sol = Solution()
for n, dislikes, expected in examples:
    got = sol.possibleBipartition(n, dislikes)
    print(n, dislikes, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

4 [[1, 2], [1, 3], [2, 4]] -> True OK
3 [[1, 2], [1, 3], [2, 3]] -> False OK


## 10. Vì sao đúng

**(a) Tương đương với bài toán tô hai màu.** Gán mỗi người một nhóm (màu); điều kiện của đề là mọi cạnh có hai đầu khác nhóm, đúng là định nghĩa tô hai màu hợp lệ.

**(b) Tính đúng của lan truyền (Cách 2, 3).** Trong một thành phần liên thông, nếu có một cách chia hợp lệ thì đổi nhóm của **cả** thành phần cũng hợp lệ; do đó có thể giả sử đỉnh xuất phát màu 1. Mỗi láng giềng của đỉnh màu `c` **bắt buộc** màu `3 - c`; theo quy nạp theo khoảng cách từ gốc, màu của mọi đỉnh trong thành phần bị ép duy nhất. Thuật toán tô đúng màu bị ép đó, và nếu một cạnh nối hai đỉnh cùng màu thì không thể có cách chia hợp lệ nào (vì cách duy nhất có thể vừa bị vi phạm). Nếu không có xung đột thì tô được mọi cạnh khác màu, tức là một cách chia hợp lệ. Các thành phần độc lập nhau nên kết quả toàn cục là "và" các thành phần.

**(c) Cách 4 (Union Find).** Gọi `~` là quan hệ tương đương nhỏ nhất thỏa: hai kẻ thù của cùng một người thì `~` nhau. Trong **mọi** cách chia hợp lệ, hai người cùng bị một người `x` ghét đều khác nhóm với `x`, tức là **cùng nhóm** với nhau; do đó mọi cặp `~` cùng nhóm. Vì vậy:

- Nếu hai đầu `x`, `y` của một cạnh nằm trong cùng lớp `~` thì chúng bị ép cùng nhóm nhưng lại ghét nhau ⇒ **không chia được**.
- Ngược lại, nếu không cạnh nào có hai đầu cùng lớp, thì chia được: mọi kẻ thù của một đỉnh nằm trong cùng một lớp `L(x)` và `x` không thuộc lớp đó; tô các lớp thành hai nhóm theo cấu trúc hai phía của đồ thị "lớp - lớp" (nối `L(x)` với lớp chứa `x`) hợp lệ.

Thuật toán xử lý các đỉnh `x` lần lượt: mỗi lần gộp các kẻ thù của `x` rồi kiểm tra các cạnh của `x` với trạng thái hiện tại. Lớp chỉ **lớn dần**, nên nếu có mâu thuẫn ở một thời điểm thì cũng có ở trạng thái cuối; ngược lại, nếu ở trạng thái cuối hai đầu `x - y` cùng lớp thì khi xử lý đỉnh đến sau của hai đỉnh gộp lớp, hoặc khi xử lý `x` lẫn `y` (mỗi cạnh được kiểm tra từ cả hai đầu, sau khi các gộp do chính đầu đó sinh ra đã được thực hiện) sẽ phát hiện. Phần kiểm chứng bằng máy bên dưới xác nhận kết luận này trên hàng nghìn đồ thị.

**Kiểm chứng bằng máy.** Dựng ba nguồn đáp án **độc lập** với các lời giải: (1) vét cạn `2^n` (`n <= 14`); (2) đồ thị **sinh có chủ đích** (planted): chọn trước phân hoạch rồi chỉ thêm cạnh xuyên phân hoạch (đáp án `True` chắc chắn); thêm một chu trình lẻ (chắc chắn `False`); (3) với đáp án `True`, kiểm tra **bằng chứng**: một cách chia thật sự (tô màu mà mọi cạnh có hai đầu khác nhóm).

In [16]:
def valid_partition(n: int, dislikes: List[List[int]], groups: List[int]) -> bool:
    """groups[i] là nhóm (0/1) của người i (1..n)."""
    return len(groups) == n + 1 and all(groups[a] != groups[b] for a, b in dislikes)

def find_partition(n: int, dislikes: List[List[int]]):
    """Trả về một cách chia (list nhóm 0/1, chỉ số 1..n) hoặc None. Dùng BFS, độc lập với bản nộp."""
    g = build_graph(n, dislikes)
    grp = [-1] * (n + 1)
    for s in range(1, n + 1):
        if grp[s] != -1:
            continue
        grp[s] = 0
        dq = deque([s])
        while dq:
            u = dq.popleft()
            for v in g[u]:
                if grp[v] == -1:
                    grp[v] = 1 - grp[u]
                    dq.append(v)
                elif grp[v] == grp[u]:
                    return None
    return grp

def all_pairs(n):
    return [[a, b] for a in range(1, n + 1) for b in range(a + 1, n + 1)]

def random_dislikes(rng, n, m):
    pairs = all_pairs(n)
    m = min(m, len(pairs))
    return sorted(rng.sample(pairs, m))

def planted(rng, n, m, make_false):
    """Đồ thị có đáp án biết trước: True nếu mọi cạnh xuyên phân hoạch; False nếu thêm một chu trình lẻ."""
    side = [rng.randint(0, 1) for _ in range(n + 1)]
    cross = [[a, b] for a in range(1, n + 1) for b in range(a + 1, n + 1) if side[a] != side[b]]
    edges = set(map(tuple, rng.sample(cross, min(m, len(cross))))) if cross else set()
    if make_false:
        L = rng.choice([x for x in (3, 5, 7, 9) if x <= n] or [3])
        if L > n:
            return None
        nodes = rng.sample(range(1, n + 1), L)
        for i in range(L):
            a, b = nodes[i], nodes[(i + 1) % L]
            edges.add((min(a, b), max(a, b)))
    return [list(e) for e in sorted(edges)]

rng = random.Random(886)
for _ in range(300):
    n = rng.randint(1, 12)
    d = random_dislikes(rng, n, rng.randint(0, min(len(all_pairs(n)), 14)))
    b = bipartition_brute(n, d)
    p = find_partition(n, d)
    assert b == (p is not None), (n, d)
    if p is not None:
        assert valid_partition(n, d, p)
print("oracle vét cạn và oracle bằng chứng (BFS) khớp nhau trên 300 đồ thị ngẫu nhiên")

oracle vét cạn và oracle bằng chứng (BFS) khớp nhau trên 300 đồ thị ngẫu nhiên


## 11. Time / Space Complexity

Gọi `n` là số người, `m = len(dislikes)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cách chia | `O(2^n * m)` | `O(n)` |
| 2. DFS tô màu | `O(n + m)` | `O(n + m)` |
| 3. BFS tô màu | `O(n + m)` | `O(n + m)` |
| 4. Union Find | `O((n + m) alpha(n))` | `O(n + m)` |

Đo thử với `n = 2000`, `m = 10^4` (cỡ tối đa), cả hai loại đáp án:

In [17]:
rng = random.Random(11)
n, m = 2000, 10000
d_true = planted(rng, n, m, False)
d_false = planted(rng, n, m, True)
assert find_partition(n, d_true) is not None and find_partition(n, d_false) is None
for label, d, expected in (("True", d_true, True), ("False", d_false, False)):
    for name, fn in (("Cách 2 DFS", sol_dfs), ("Cách 3 BFS", sol_bfs), ("Cách 4 DSU", sol_dsu)):
        t0 = time.perf_counter()
        got = fn(n, d)
        dt = (time.perf_counter() - t0) * 1000
        assert got == expected, (label, name)
        print(f"{label:5} {name:11} n={n}, m={len(d)}: {dt:7.2f} ms")

True  Cách 2 DFS  n=2000, m=10000:    2.75 ms
True  Cách 3 BFS  n=2000, m=10000:    2.54 ms
True  Cách 4 DSU  n=2000, m=10000:   10.50 ms
False Cách 2 DFS  n=2000, m=10003:    1.47 ms
False Cách 3 BFS  n=2000, m=10003:    1.79 ms
False Cách 4 DSU  n=2000, m=10003:    4.85 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc `1 <= ai < bi <= n`, các cặp duy nhất) và kiểm tra bằng chứng.

In [18]:
def solutions_list():
    return {
        "dfs": bipartition_dfs,
        "bfs": bipartition_bfs,
        "dsu": bipartition_dsu,
        "sol_dfs": sol_dfs,
        "sol_bfs": sol_bfs,
        "sol_dsu": sol_dsu,
        "final": Solution().possibleBipartition,
    }

def run_all(n, d, expected=None, brute_limit=14):
    p = find_partition(n, d)
    ref = p is not None
    if p is not None:
        assert valid_partition(n, d, p)
    if n <= brute_limit:
        assert bipartition_brute(n, d) == ref, ("oracle lệch", n, d)
    for name, fn in solutions_list().items():
        assert fn(n, [list(e) for e in d]) == ref, (name, n, d)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", n, d, expected, ref)
    return ref

# 12.1 ví dụ đề
for n, d, e in examples:
    assert run_all(n, d, e) == e
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (1, [], True),                                     # một người
    (2, [], True),                                     # không ai ghét ai
    (2, [[1, 2]], True),                               # một cạnh
    (3, [[1, 2], [2, 3]], True),                       # đường đi
    (3, [[1, 2], [1, 3], [2, 3]], False),              # tam giác
    (4, [[1, 2], [2, 3], [3, 4], [1, 4]], True),       # chu trình 4 (chẵn)
    (5, [[1, 2], [2, 3], [3, 4], [4, 5], [1, 5]], False),   # chu trình 5 (lẻ)
    (5, [[1, 2], [3, 4], [4, 5], [3, 5]], False),      # thành phần thứ hai là tam giác, thành phần đầu ổn
    (6, [[1, 2], [3, 4], [5, 6]], True),               # nhiều thành phần rời nhau
    (6, [[1, 4], [2, 5], [3, 6], [1, 5], [2, 6], [3, 4]], True),   # chu trình 6
    (4, [[1, 2], [1, 3], [1, 4], [2, 3]], False),      # có tam giác 1-2-3
    (5, [[1, 5], [2, 5], [3, 5], [4, 5]], True),       # hình sao
    (4, all_pairs(4), False),                          # đồ thị đầy đủ K4
    (3, [[2, 3]], True),                               # người 1 cô lập
]
for n, d, e in edge:
    assert run_all(n, d, e) == e, (n, d)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_only_from_one(n, dislikes):             # chỉ tô màu từ người 1 (bỏ sót các thành phần khác)
    g = build_graph(n, dislikes)
    color = [0] * (n + 1)
    color[1] = 1
    stack = [1]
    while stack:
        u = stack.pop()
        for v in g[u]:
            if color[v] == 0:
                color[v] = 3 - color[u]
                stack.append(v)
            elif color[v] == color[u]:
                return False
    return True

def wrong_no_conflict_check(n, dislikes):          # chỉ tô màu, không bao giờ so sánh với đỉnh đã tô
    g = build_graph(n, dislikes)
    color = [0] * (n + 1)
    for s in range(1, n + 1):
        if color[s]:
            continue
        color[s] = 1
        stack = [s]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if color[v] == 0:
                    color[v] = 3 - color[u]
                    stack.append(v)
    return True

def wrong_visited_only(n, dislikes):               # dùng tập đã thăm, bỏ qua màu: coi đã thăm là ổn
    g = build_graph(n, dislikes)
    seen = set()
    for s in range(1, n + 1):
        if s in seen:
            continue
        seen.add(s)
        stack = [s]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
    return True

def wrong_only_triangles(n, dislikes):             # chỉ phát hiện tam giác, bỏ sót chu trình lẻ dài hơn
    g = build_graph(n, dislikes)
    sets = [set(x) for x in g]
    for a, b in dislikes:
        if sets[a] & sets[b]:
            return False
    return True

def find_counter(wrong, tries=20000, seed=3):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 8)
        d = random_dislikes(rng, n, rng.randint(1, 9))
        if wrong(n, d) != bipartition_brute(n, d):
            return n, d
    return None

counters = {}
for name, fn in (("chỉ tô từ người 1", wrong_only_from_one), ("không kiểm tra xung đột", wrong_no_conflict_check),
                 ("chỉ đánh dấu đã thăm", wrong_visited_only), ("chỉ tìm tam giác", wrong_only_triangles)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    n, d = c
    print(f"{name:26} n={n} dislikes={d}: sai = {fn(n, d)}, đúng = {bipartition_brute(n, d)}")
    assert run_all(n, d) == bipartition_brute(n, d)
print("12.3 phản ví dụ: OK")

chỉ tô từ người 1          n=7 dislikes=[[1, 3], [2, 4], [2, 5], [2, 6], [4, 6], [4, 7]]: sai = True, đúng = False
không kiểm tra xung đột    n=3 dislikes=[[1, 2], [1, 3], [2, 3]]: sai = True, đúng = False
chỉ đánh dấu đã thăm       n=3 dislikes=[[1, 2], [1, 3], [2, 3]]: sai = True, đúng = False
chỉ tìm tam giác           n=7 dislikes=[[1, 5], [1, 6], [2, 3], [2, 4], [2, 5], [3, 6], [3, 7], [4, 6], [5, 7]]: sai = True, đúng = False
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ: so mọi cách với oracle vét cạn (n <= 12) và oracle bằng chứng
rng = random.Random(4886)
count_true = 0
for _ in range(1500):
    n = rng.randint(1, 12)
    d = random_dislikes(rng, n, rng.randint(0, min(len(all_pairs(n)), 16)))
    count_true += run_all(n, d, None, brute_limit=12)
print(f"12.4 ngẫu nhiên n <= 12: OK ({count_true}/1500 đồ thị chia được)")

# 12.5 đồ thị sinh có chủ đích (planted): biết trước đáp án, cỡ trung bình
rng = random.Random(5886)
for _ in range(300):
    n = rng.randint(3, 80)
    m = rng.randint(0, 150)
    for make_false in (False, True):
        d = planted(rng, n, m, make_false)
        if d is None:
            continue
        got = run_all(n, d, None, brute_limit=0)
        assert got == (not make_false), (n, d, make_false)
print("12.5 đồ thị planted (đáp án biết trước): OK")

# 12.6 lớn n = 2000, m = 10^4 (nhiều dạng)
rng = random.Random(6886)
shapes = {
    "planted True": (2000, planted(rng, 2000, 10000, False)),
    "planted False": (2000, planted(rng, 2000, 9990, True)),
    "đường đi dài": (2000, [[i, i + 1] for i in range(1, 2000)]),
    "chu trình chẵn": (2000, [[i, i + 1] for i in range(1, 2000)] + [[1, 2000]]),
    "đường đi + một cạnh lẻ": (2000, [[i, i + 1] for i in range(1, 2000)] + [[1, 3]]),
    "không cạnh": (2000, []),
}
expected_shape = {"planted True": True, "planted False": False, "đường đi dài": True, "chu trình chẵn": True,
                  "đường đi + một cạnh lẻ": False, "không cạnh": True}
for name, (n, d) in shapes.items():
    assert len(d) <= 10 ** 4
    got = run_all(n, d, expected_shape[name], brute_limit=0)
    print(f"  {name:24} n={n}, m={len(d)}: {got}")
print("12.6 n = 2000: OK")

# 12.7 tính chất: kết quả không đổi khi hoán vị nhãn; thêm cạnh không biến False thành True
rng = random.Random(7886)
for _ in range(300):
    n = rng.randint(2, 12)
    d = random_dislikes(rng, n, rng.randint(0, 15))
    perm = list(range(1, n + 1))
    rng.shuffle(perm)
    mapped = sorted([min(perm[a - 1], perm[b - 1]), max(perm[a - 1], perm[b - 1])] for a, b in d)
    assert Solution().possibleBipartition(n, d) == Solution().possibleBipartition(n, mapped)
    if not Solution().possibleBipartition(n, d):
        extra = random_dislikes(rng, n, 3)
        merged = sorted(map(list, {tuple(e) for e in d + extra}))
        assert Solution().possibleBipartition(n, merged) is False
print("12.7 bất biến theo hoán vị nhãn / đơn điệu khi thêm cạnh: OK")

12.4 ngẫu nhiên n <= 12: OK (879/1500 đồ thị chia được)
12.5 đồ thị planted (đáp án biết trước): OK


  planted True             n=2000, m=10000: True
  planted False            n=2000, m=9999: False
  đường đi dài             n=2000, m=1999: True
  chu trình chẵn           n=2000, m=2000: True
  đường đi + một cạnh lẻ   n=2000, m=2000: False
  không cạnh               n=2000, m=0: True
12.6 n = 2000: OK
12.7 bất biến theo hoán vị nhãn / đơn điệu khi thêm cạnh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên rằng nhãn bắt đầu từ `1` (mảng cỡ `n` thay vì `n + 1`)

Tạo `g = [[] for _ in range(n)]` rồi truy cập `g[n]` gây `IndexError`; hoặc lệch chỉ số nên bỏ sót người.

In [21]:
def wrong_zero_indexed(n, dislikes):
    g = [[] for _ in range(n)]            # sai: chỉ có chỉ số 0..n-1
    for a, b in dislikes:
        g[a].append(b)
        g[b].append(a)
    return True

try:
    wrong_zero_indexed(4, [[1, 2], [1, 3], [2, 4]])
    print("không lỗi?")
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().possibleBipartition(4, [[1, 2], [1, 3], [2, 4]]))

IndexError: list index out of range
đúng: True


Vì sao sai: người `n` không có chỗ trong mảng cỡ `n`. **Đúng:** dùng `n + 1` phần tử (bỏ phần tử `0`) hoặc trừ 1 đồng nhất mọi nơi.

### Lỗi 2 — Chỉ tô màu từ người 1 (bỏ sót các thành phần khác)

Đồ thị có thể không liên thông; thành phần chứa chu trình lẻ có thể nằm **xa** người 1.

In [22]:
n, d = counters["chỉ tô từ người 1"]
print("n =", n, "dislikes =", d, "| sai ->", wrong_only_from_one(n, d), "| đúng ->", Solution().possibleBipartition(n, d))
assert wrong_only_from_one(n, d) is True and Solution().possibleBipartition(n, d) is False
print("ví dụ gọn:", wrong_only_from_one(5, [[1, 2], [3, 4], [4, 5], [3, 5]]), "(sai) so với", Solution().possibleBipartition(5, [[1, 2], [3, 4], [4, 5], [3, 5]]))

n = 7 dislikes = [[1, 3], [2, 4], [2, 5], [2, 6], [4, 6], [4, 7]] | sai -> True | đúng -> False
ví dụ gọn: True (sai) so với False


Vì sao sai: tam giác `3-4-5` không liên thông với `1`. **Đúng:** vòng `for s in range(1, n + 1)`: nếu `s` chưa tô thì bắt đầu một lần tô mới.

### Lỗi 3 — Tô màu nhưng không kiểm tra xung đột

Chỉ tô màu cho đỉnh chưa tô mà không bao giờ so sánh màu hai đầu một cạnh đã tô: luôn trả `True`.

In [23]:
n, d = counters["không kiểm tra xung đột"]
print("n =", n, "dislikes =", d, "| sai ->", wrong_no_conflict_check(n, d), "| đúng ->", Solution().possibleBipartition(n, d))
assert wrong_no_conflict_check(n, d) is True and Solution().possibleBipartition(n, d) is False

n = 3 dislikes = [[1, 2], [1, 3], [2, 3]] | sai -> True | đúng -> False


Vì sao sai: mâu thuẫn chỉ lộ ra khi gặp một đỉnh **đã tô** và so màu. **Đúng:** thêm nhánh `elif color[v] == color[u]: return False`.

### Lỗi 4 — Chỉ dùng "đã thăm" thay cho màu

Tập `seen` chỉ biết đã đi qua một đỉnh, không biết nó thuộc nhóm nào, nên không phát hiện được chu trình lẻ.

In [24]:
n, d = counters["chỉ đánh dấu đã thăm"]
print("n =", n, "dislikes =", d, "| sai ->", wrong_visited_only(n, d), "| đúng ->", Solution().possibleBipartition(n, d))
assert wrong_visited_only(n, d) is True and Solution().possibleBipartition(n, d) is False

n = 3 dislikes = [[1, 2], [1, 3], [2, 3]] | sai -> True | đúng -> False


Vì sao sai: duyệt đồ thị đã thăm không có thông tin phân nhóm. **Đúng:** lưu **màu** (hoặc độ chẵn lẻ của khoảng cách) cho mỗi đỉnh.

### Lỗi 5 — Chỉ tìm tam giác

Chu trình lẻ có thể dài hơn 3 (5, 7, ...). Chỉ kiểm tra "hai đầu cạnh có láng giềng chung" bỏ sót chúng.

In [25]:
n, d = counters["chỉ tìm tam giác"]
print("n =", n, "dislikes =", d, "| sai ->", wrong_only_triangles(n, d), "| đúng ->", Solution().possibleBipartition(n, d))
assert wrong_only_triangles(n, d) is True and Solution().possibleBipartition(n, d) is False
print("chu trình 5:", wrong_only_triangles(5, [[1, 2], [2, 3], [3, 4], [4, 5], [1, 5]]), "(sai) so với", Solution().possibleBipartition(5, [[1, 2], [2, 3], [3, 4], [4, 5], [1, 5]]))

n = 7 dislikes = [[1, 5], [1, 6], [2, 3], [2, 4], [2, 5], [3, 6], [3, 7], [4, 6], [5, 7]] | sai -> True | đúng -> False
chu trình 5: True (sai) so với False


Vì sao sai: chu trình 5 không có tam giác nhưng vẫn là chu trình lẻ. **Đúng:** tô hai màu (hoặc Union Find) phát hiện mọi chu trình lẻ.

### Lỗi 6 — Đệ quy DFS quá sâu

Đệ quy trên đường đi `2000` đỉnh vượt giới hạn mặc định `1000` của Python và gây `RecursionError`.

In [26]:
def recursive_dfs_colorable(n, dislikes):
    g = build_graph(n, dislikes)
    color = [0] * (n + 1)

    def dfs(u, c):
        color[u] = c
        for v in g[u]:
            if color[v] == c:
                return False
            if color[v] == 0 and not dfs(v, 3 - c):
                return False
        return True

    return all(color[s] != 0 or dfs(s, 1) for s in range(1, n + 1))

path_n = 2000
path = [[i, i + 1] for i in range(1, path_n)]
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    recursive_dfs_colorable(path_n, path)
    print("không lỗi (giới hạn đủ lớn)")
except RecursionError as e:
    print("RecursionError:", e)
finally:
    sys.setrecursionlimit(old_limit)
print("bản dùng ngăn xếp tường minh:", Solution().possibleBipartition(path_n, path))

RecursionError: maximum recursion depth exceeded
bản dùng ngăn xếp tường minh: True


Vì sao sai: mỗi đỉnh trên đường đi thêm một khung gọi hàm. **Đúng:** dùng ngăn xếp/hàng đợi tường minh (như bản nộp), hoặc tăng `sys.setrecursionlimit` (kém an toàn).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia thành hai nhóm sao cho các cặp bị cấm không cùng nhóm" ⇒ **kiểm tra đồ thị hai phía** (tô hai màu / chu trình lẻ).
- **Khung chuẩn:** danh sách kề + mảng màu `0/1/2` + vòng `for` qua mọi đỉnh (đồ thị không liên thông) + BFS/DFS.
- Đổi màu bằng `3 - c` (màu `1 <-> 2`) hoặc `1 - c` (màu `0 <-> 1`, khi dùng `-1` làm chưa tô).
- Nhãn `1..n` ⇒ mảng cỡ `n + 1`.
- Union Find thay thế: tất cả kẻ thù của một người cùng nhóm; hoặc Union Find có tính chẵn lẻ (parity).
- Cùng họ: Is Graph Bipartite? (785), Course Schedule (207), Number of Provinces (547), Redundant Connection (684).
- Test nên có: `n = 1`, không cạnh, tam giác, chu trình 4 và 5, thành phần rời nhau trong đó chỉ thành phần xa người 1 vi phạm, đường đi dài `n = 2000`.

## 15. Biến thể

1. **Trả về chính cách chia** (không chỉ đúng/sai): đã có `find_partition` ở mục 10.
2. **Đếm số cách chia** thành hai nhóm có nhãn (bài tập 5): `2^c` với `c` là số thành phần liên thông nếu chia được, ngược lại `0`.
3. **Chia thành `k` nhóm** (tô `k` màu): NP-khó khi `k >= 3`.
4. Bài liên quan: Is Graph Bipartite? (785), Flower Planting With No Adjacent (1042).

Dưới đây là code biến thể 2, kiểm bằng vét cạn.

In [27]:
def count_partitions(n: int, dislikes: List[List[int]]) -> int:
    g = build_graph(n, dislikes)
    color = [0] * (n + 1)
    comps = 0
    for s in range(1, n + 1):
        if color[s]:
            continue
        comps += 1
        color[s] = 1
        stack = [s]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if color[v] == 0:
                    color[v] = 3 - color[u]
                    stack.append(v)
                elif color[v] == color[u]:
                    return 0
    return 2 ** comps

def count_brute(n, dislikes):
    return sum(1 for assign in product((0, 1), repeat=n) if all(assign[a - 1] != assign[b - 1] for a, b in dislikes))

rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 10)
    d = random_dislikes(rng, n, rng.randint(0, 12))
    assert count_partitions(n, d) == count_brute(n, d), (n, d)
print("ví dụ 1:", count_partitions(*examples[0][:2]), "| ví dụ 2:", count_partitions(*examples[1][:2]))
assert count_partitions(*examples[0][:2]) == 2 and count_partitions(*examples[1][:2]) == 0

ví dụ 1: 2 | ví dụ 2: 0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cách chia | Định nghĩa | `O(2^n m)` | `O(n)` | Oracle, `n <= 14` |
| 2. DFS tô màu | Lan truyền màu bằng ngăn xếp | `O(n + m)` | `O(n + m)` | **Cách nộp chính** |
| 3. BFS tô màu | Lan truyền theo lớp | `O(n + m)` | `O(n + m)` | Khi cần khoảng cách/chu trình lẻ ngắn nhất |
| 4. Union Find | Kẻ thù của kẻ thù là bạn | `O((n + m) alpha)` | `O(n + m)` | Cạnh đến dần (online), không cần duyệt đồ thị |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
g = build_graph(4, [[1, 2], [1, 3], [2, 4]])
print(g[1], g[4])
print(3 - 1, 3 - 2)
print(bipartition_brute(3, [[1, 2], [2, 3]]))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 6`, `dislikes = [[1,2],[2,3],[4,5],[5,6],[4,6]]`: có chia được không? Nếu không, ghi ba người tạo chu trình lẻ.

In [29]:
# Bài 2: điền kết quả tính tay
hand_possible = None          # True / False
hand_odd_cycle = None         # ví dụ [a, b, c] (nếu không chia được)

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành tô màu bằng BFS.

In [30]:
def exercise_3_bfs(n: int, dislikes: List[List[int]]) -> bool:
    g = [[] for _ in range(n + 1)]
    for a, b in dislikes:
        g[a].append(b)
        g[b].append(a)
    color = [0] * (n + 1)
    for s in range(1, n + 1):
        if color[s]:
            continue
        color[s] = 1
        dq = deque([s])
        while dq:
            u = dq.popleft()
            for v in g[u]:
                # TODO 1: nếu color[v] == color[u] thì trả False
                # TODO 2: nếu v chưa tô thì tô màu 3 - color[u] và đưa v vào hàng đợi
                pass
    return True


# for n, d, e in examples:
#     assert exercise_3_bfs(n, d) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_dsu(n, dislikes)` bằng Union Find "kẻ thù của kẻ thù là bạn".

In [31]:
def exercise_4_dsu(n: int, dislikes: List[List[int]]) -> bool:
    # TODO
    pass


# for n, d, e in examples:
#     assert exercise_4_dsu(n, d) == e

### Bài 5 — Mini challenge: đếm số cách chia (khó)

Viết `exercise_5_count(n, dislikes)` trả về số cách gán mỗi người vào nhóm `0` hoặc `1` (nhóm **có nhãn**, cho phép nhóm rỗng) sao cho không cặp ghét nhau nào cùng nhóm.

```python
exercise_5_count(4, [[1,2],[1,3],[2,4]])      -> 2
exercise_5_count(3, [[1,2],[1,3],[2,3]])      -> 0
exercise_5_count(3, [])                       -> 8
exercise_5_count(4, [[1,2],[3,4]])            -> 4
```
Gợi ý: nếu chia được thì mỗi thành phần liên thông có đúng 2 cách tô.

In [32]:
def exercise_5_count(n: int, dislikes: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_count(4, [[1, 2], [1, 3], [2, 4]]) == 2
# assert exercise_5_count(3, [[1, 2], [1, 3], [2, 3]]) == 0
# assert exercise_5_count(3, []) == 8
# assert exercise_5_count(4, [[1, 2], [3, 4]]) == 4

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `g[i]` là danh sách láng giềng của `i`.
- Mức 2: `g[1]` gồm `2, 3` (theo thứ tự cạnh); `g[4]` chỉ có `2`.
- Mức 3: `[2, 3] [2]`, `2 1`, `True`.

**Bài 2**
- Mức 1: tách đồ thị thành hai thành phần `{1,2,3}` và `{4,5,6}`.
- Mức 2: `{1,2,3}` là đường đi (chia được); `{4,5,6}` có cạnh `4-5`, `5-6`, `4-6`.
- Mức 3: `False`, chu trình lẻ `[4, 5, 6]`.

**Bài 3**
- Mức 1: so màu trước khi tô để phát hiện xung đột với đỉnh đã tô.
- Mức 2: `if color[v] == color[u]: return False`.
- Mức 3: `if color[v] == 0: color[v] = 3 - color[u]; dq.append(v)`.

**Bài 4**
- Mức 1: `find` có nén đường; `parent = list(range(n + 1))`.
- Mức 2: với mỗi `x` và mỗi kẻ thù `y`: nếu `find(x) == find(y)` trả `False`.
- Mức 3: gộp `y` vào tập của kẻ thù đầu tiên: `parent[find(y)] = find(g[x][0])`.

**Bài 5**
- Mức 1: đếm số thành phần liên thông `c`.
- Mức 2: mỗi thành phần có 2 cách tô nếu không có chu trình lẻ.
- Mức 3: trả `0` nếu có xung đột, ngược lại `2 ** c`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
g = build_graph(4, [[1, 2], [1, 3], [2, 4]])
print(g[1], g[4])
print(3 - 1, 3 - 2)
print(bipartition_brute(3, [[1, 2], [2, 3]]))

assert (g[1], g[4]) == ([2, 3], [2])
assert (3 - 1, 3 - 2) == (2, 1)
assert bipartition_brute(3, [[1, 2], [2, 3]]) is True

[2, 3] [2]
2 1
True


### Đáp án Bài 2

`{1,2,3}` là đường đi nên chia được, nhưng `{4,5,6}` là tam giác: `False`, chu trình lẻ `[4, 5, 6]`.

In [34]:
n, d = 6, [[1, 2], [2, 3], [4, 5], [5, 6], [4, 6]]
hand_possible_solution = False
hand_odd_cycle_solution = [4, 5, 6]
assert Solution().possibleBipartition(n, d) is hand_possible_solution
assert bipartition_brute(n, d) is hand_possible_solution
a, b, c = hand_odd_cycle_solution
edges = {tuple(e) for e in d}
assert {(a, b), (b, c), (a, c)} <= edges
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_bfs_solution(n: int, dislikes: List[List[int]]) -> bool:
    g = [[] for _ in range(n + 1)]
    for a, b in dislikes:
        g[a].append(b)
        g[b].append(a)
    color = [0] * (n + 1)
    for s in range(1, n + 1):
        if color[s]:
            continue
        color[s] = 1
        dq = deque([s])
        while dq:
            u = dq.popleft()
            for v in g[u]:
                if color[v] == color[u]:
                    return False
                if color[v] == 0:
                    color[v] = 3 - color[u]
                    dq.append(v)
    return True

for n, d, e in examples:
    assert exercise_3_bfs_solution(n, d) == e
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 10)
    d = random_dislikes(rng, n, rng.randint(0, 12))
    assert exercise_3_bfs_solution(n, d) == bipartition_brute(n, d)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_dsu_solution(n: int, dislikes: List[List[int]]) -> bool:
    g = [[] for _ in range(n + 1)]
    for a, b in dislikes:
        g[a].append(b)
        g[b].append(a)
    parent = list(range(n + 1))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for x in range(1, n + 1):
        for y in g[x]:
            if find(x) == find(y):
                return False
            parent[find(y)] = find(g[x][0])
    return True

for n, d, e in examples:
    assert exercise_4_dsu_solution(n, d) == e
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 10)
    d = random_dislikes(rng, n, rng.randint(0, 12))
    assert exercise_4_dsu_solution(n, d) == bipartition_brute(n, d)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_count_solution(n: int, dislikes: List[List[int]]) -> int:
    g = [[] for _ in range(n + 1)]
    for a, b in dislikes:
        g[a].append(b)
        g[b].append(a)
    color = [0] * (n + 1)
    comps = 0
    for s in range(1, n + 1):
        if color[s]:
            continue
        comps += 1
        color[s] = 1
        stack = [s]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if color[v] == 0:
                    color[v] = 3 - color[u]
                    stack.append(v)
                elif color[v] == color[u]:
                    return 0
    return 2 ** comps

assert exercise_5_count_solution(4, [[1, 2], [1, 3], [2, 4]]) == 2
assert exercise_5_count_solution(3, [[1, 2], [1, 3], [2, 3]]) == 0
assert exercise_5_count_solution(3, []) == 8
assert exercise_5_count_solution(4, [[1, 2], [3, 4]]) == 4
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 10)
    d = random_dislikes(rng, n, rng.randint(0, 12))
    assert exercise_5_count_solution(n, d) == count_brute(n, d), (n, d)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tô hai màu bằng ngăn xếp tường minh)

```python
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        g = [[] for _ in range(n + 1)]                  # nhãn 1..n -> cỡ n + 1
        for a, b in dislikes:
            g[a].append(b)
            g[b].append(a)
        color = [0] * (n + 1)                            # 0 chưa tô, 1 / 2 hai nhóm
        for s in range(1, n + 1):                        # mọi thành phần liên thông
            if color[s]:
                continue
            color[s] = 1
            stack = [s]
            while stack:
                u = stack.pop()
                for v in g[u]:
                    if color[v] == 0:
                        color[v] = 3 - color[u]
                        stack.append(v)
                    elif color[v] == color[u]:           # hai đầu một cạnh cùng màu
                        return False
        return True
```

Công thức nhớ nhanh: hai phía ⇔ không có chu trình lẻ ⇔ tô được hai màu; láng giềng đổi màu `3 - c`; chạy từ mọi đỉnh chưa tô.

### Checklist trước khi submit

- [ ] Mảng cỡ `n + 1` (nhãn bắt đầu từ `1`)?
- [ ] Vòng lặp ngoài qua **mọi** đỉnh để phủ các thành phần liên thông?
- [ ] Có nhánh kiểm tra xung đột `color[v] == color[u]`?
- [ ] Không dùng đệ quy sâu (hoặc đã xử lý giới hạn đệ quy)?
- [ ] Đã thử: `n = 1`, không cạnh, tam giác, chu trình 5, thành phần xa người 1 vi phạm, đường đi `n = 2000`?
- [ ] Tên method đúng `possibleBipartition(self, n, dislikes)` và trả `bool`?